In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = [12, 8]


In [ ]:
# Load the dataset
data_dir = Path('DATASET_LOGS/batch_20250710_234233_app-smart-manufacturing')

# Load individual experiments
experiments = []
for exp_file in data_dir.glob('exp_*.json'):
    with open(exp_file) as f:
        exp_data = json.load(f)
        experiments.append(exp_data)

In [ ]:
# Extract tool calls from agent actions
def extract_tool_calls(agent_actions):
    tool_calls = []
    for action in agent_actions:
        if action['type'] == 'sleep':
            tool_calls.append('sleep')
            continue       
        
        tool_calls.append(action['function'])
    return tool_calls

# Create a comprehensive DataFrame
data = []
for exp in experiments:
    row = {
        'experiment_id': exp['experiment_id'],
        'chaos_mode': exp['chaos_config']['mode'],
        'fault_type': exp['chaos_config']['fault_type'],
        'intensity': exp['chaos_config']['intensity'],
        'success': exp['experiment_success'],
        'agent_success': exp['agent_success'],
        'recovery_successful': exp['recovery_successful'],
        'duration_seconds': exp['duration_seconds'],
        'agent_duration_seconds': exp['agent_duration_seconds'],
        'final_success_rate': exp['final_success_rate'],
        'final_latency_p95': exp['final_latency_p95'],
        'baseline_latency': exp['baseline_metrics']['p95_latency_seconds'],
        'chaos_latency': exp['chaos_metrics']['p95_latency_seconds'],
        'recovery_latency': exp['recovery_metrics']['p95_latency_seconds'],
        'baseline_success_rate': exp['baseline_metrics']['success_rate_percent'],
        'chaos_success_rate': exp['chaos_metrics']['success_rate_percent'],
        'recovery_success_rate': exp['recovery_metrics']['success_rate_percent'],
        'tool_calls': extract_tool_calls(exp['agent_actions']),
        'num_tool_calls': len(extract_tool_calls(exp['agent_actions'])),
        'chaos_config': exp['chaos_config'],
        'problem_location': exp['problem_location'],
    }
    data.append(row)

df = pd.DataFrame(data)
print(f"Created DataFrame with {len(df)} experiments and {len(df.columns)} features")
print("\\nChaos types:", df['fault_type'].unique())
print("\\nSample data:")
df.head()


In [ ]:
# Collect all tool calls
all_tool_calls = []
for tool_list in df['tool_calls']:
    all_tool_calls.extend(tool_list)

tool_counts = Counter(all_tool_calls)
print(f"Total tool calls across all experiments: {len(all_tool_calls)}")
print(f"Unique tools used: {len(tool_counts)}")
print("\\nTool usage:")
for tool, count in tool_counts.most_common():
    print(f"  {tool}: {count}")


In [ ]:
# Create improved pie chart with "Other" category for tools < 1%
tools = list(tool_counts.keys())
counts = list(tool_counts.values())

# Calculate total for percentage calculation
total_calls = sum(counts)

# Separate tools into main categories and "Other"
main_tools = []
main_counts = []
other_count = 0

for tool, count in zip(tools, counts):
    percentage = (count / total_calls) * 100
    if percentage >= 1.0:  # Tools with 1% or more
        main_tools.append(tool)
        main_counts.append(count)
    else:  # Tools with less than 1%
        other_count += count

# Add "Other" category if there are any small tools
if other_count > 0:
    main_tools.append("Other")
    main_counts.append(other_count)

print(f"Grouped {len(tools) - len(main_tools) + (1 if other_count > 0 else 0)} tools into 'Other' category")
print(f"Other category represents {(other_count/total_calls)*100:.1f}% of total calls")

# Create figure with larger size for better visibility
plt.figure(figsize=(18, 14))

# Define a more distinct color palette with better contrast
import matplotlib.colors as mcolors

# Use a combination of different color palettes for maximum distinction
color_list = [
    '#FF6B6B',  # Red
    '#4ECDC4',  # Teal
    '#45B7D1',  # Blue
    '#96CEB4',  # Green
    '#FECA57',  # Yellow
    '#FF9FF3',  # Pink
    '#54A0FF',  # Light Blue
    '#5F27CD',  # Purple
    '#00D2D3',  # Cyan
    '#FF9F43',  # Orange
    '#10AC84',  # Dark Green
    '#EE5A24',  # Dark Orange
    '#0984E3',  # Dark Blue
    '#6C5CE7',  # Light Purple
    '#FD79A8'   # Light Pink
]

# Use the custom colors, cycling if we have more tools than colors
# Use gray for "Other" category
colors = []
for i, tool in enumerate(main_tools):
    if tool == "Other":
        colors.append('#808080')  # Gray for Other
    else:
        colors.append(color_list[i % len(color_list)])

# Create the pie chart with much larger text and no gaps
wedges, texts, autotexts = plt.pie(main_counts, labels=None, autopct='%1.1f%%', 
                                  startangle=90, colors=colors, 
                                  textprops={'fontsize': 18, 'fontweight': 'bold'},
                                  pctdistance=0.75)  # Move percentages closer to center

# Enhance percentage text visibility (inside the pie) with much larger text
for autotext in autotexts:
    autotext.set_color('white')  # White text for better contrast on colored background
    autotext.set_fontweight('bold')
    autotext.set_fontsize(18)  # Much larger text
    autotext.set_bbox(dict(boxstyle="round,pad=0.4", facecolor='black', alpha=0.8))  # Larger background box

# Create a legend with tool names outside the pie with larger text
plt.legend(wedges, main_tools, title="Tools", loc="center left", bbox_to_anchor=(1.05, 0, 0.5, 1), 
          fontsize=15, title_fontsize=17, frameon=True, fancybox=True, shadow=True)

plt.title('Distribution of Tool Calls', fontsize=22, fontweight='bold', pad=30)
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
plt.tight_layout()
plt.show()


In [ ]:
# Create a mapping for better labels
fault_type_labels = {
    'pod_failure': 'Pod Failure',
    'memory_stress': 'Memory Stress', 
    'cpu_stress': 'CPU Stress',
    'docker_constraints': 'Node Failure (Docker)',
    'none': 'No Fault'
}

df['fault_type_label'] = df['fault_type'].map(fault_type_labels)

# Calculate success rates by chaos type
success_by_chaos = df.groupby('fault_type_label').agg({
    'success': ['count', 'sum'],
    'agent_success': 'sum',
    'recovery_successful': 'sum'
}).round(2)

success_by_chaos.columns = ['Total_Experiments', 'Experiment_Success', 'Agent_Success', 'Recovery_Success']
success_by_chaos['Experiment_Success_Rate'] = (success_by_chaos['Experiment_Success'] / success_by_chaos['Total_Experiments'] * 100).round(1)
success_by_chaos['Agent_Success_Rate'] = (success_by_chaos['Agent_Success'] / success_by_chaos['Total_Experiments'] * 100).round(1)
success_by_chaos['Recovery_Success_Rate'] = (success_by_chaos['Recovery_Success'] / success_by_chaos['Total_Experiments'] * 100).round(1)

print("Success rates by chaos type:")
print(success_by_chaos)


In [ ]:
# Create bar plot for experiment success rates only
plt.figure(figsize=(12, 8))

bars = plt.bar(success_by_chaos.index, success_by_chaos['Experiment_Success_Rate'], 
               color='lightblue', edgecolor='navy', linewidth=2)
plt.title('Experiment Success Rate by Chaos Type', fontsize=16, fontweight='bold')
plt.ylabel('Success Rate (%)', fontsize=14)
plt.ylim(0, 105)
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{height}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

def analyze_problem_location_accuracy(df):
    """
    Analyzes if the agent correctly identified the problem location, assuming
    all necessary data exists in the input DataFrame's rows.
    """
    results = []

    for _, row in df.iterrows():
        # All data is pulled directly from the current row
        chaos_config = row['chaos_config']
        problem_location_data = row.get('problem_location')
        mode = chaos_config.get('mode')

        # Extract the names of locations the agent identified
        agent_locations = []
        if isinstance(problem_location_data, list):
            agent_locations = [loc.get('name') for loc in problem_location_data if loc.get('name')]

        # 1. Determine the ground truth target from the chaos configuration
        ground_truth_target = None
        if mode == 'chaos_only':
            app = chaos_config.get('target_app')
            # The target is the node for 'node-wide' chaos, otherwise it's the app
            ground_truth_target = None if app == 'all_pods_node' else app
        elif mode == 'docker_only':
            # The target is the node where docker constraints were applied
            ground_truth_target = chaos_config.get('node_name')

        # 2. Determine if the agent's identification was correct
        correctness = False
        if mode == "no_failure":
            # For "no fault" tests, success means the agent found no locations.
            correctness = not agent_locations
        elif ground_truth_target:
            # For all fault tests, success means the agent identified the correct target.
            # This works for both app names and node names.
            correctness = any(ground_truth_target in loc for loc in agent_locations)

        results.append({
            'experiment_id': row['experiment_id'],
            'fault_type': row['chaos_config']['fault_type'],
            'ground_truth': ground_truth_target or 'None',
            'agent_locations': agent_locations if agent_locations else 'None',
            'correct_identification': correctness
        })

    return pd.DataFrame(results)

# --- Main Analysis Logic ---

# 1. Create the identification analysis DataFrame
identification_df = analyze_problem_location_accuracy(df)

# 2. Calculate accuracy by chaos type
accuracy_by_chaos = identification_df.groupby('fault_type').agg(
    Total_Experiments=('correct_identification', 'count'),
    Correct_Identifications=('correct_identification', 'sum')
).astype(int)

# 3. Calculate the accuracy rate as a percentage
accuracy_by_chaos['Accuracy_Percentage'] = (
    (accuracy_by_chaos['Correct_Identifications'] / accuracy_by_chaos['Total_Experiments']) * 100
).round(1)

print("Problem Location Identification Accuracy by Chaos Type:")
print(accuracy_by_chaos)

print("\n--- Detailed Breakdown ---")
for fault_type in identification_df['fault_type_label'].unique():
    subset = identification_df[identification_df['fault_type_label'] == fault_type]
    correct = subset['correct_identification'].sum()
    total = len(subset)
    if total > 0:
        print(f"{fault_type}: {correct}/{total} ({(correct/total*100):.1f}%)")

In [ ]:
# Create visualization for problem location identification accuracy
plt.figure(figsize=(12, 8))

chaos_types = accuracy_by_chaos.index
accuracy_percentages = accuracy_by_chaos['Accuracy_Percentage']

# Color bars based on accuracy - green for high accuracy, red for low
colors = ['green' if acc >= 70 else 'orange' if acc >= 50 else 'red' for acc in accuracy_percentages]

bars = plt.bar(chaos_types, accuracy_percentages, color=colors, alpha=0.7, edgecolor='black', linewidth=2)

plt.title('Problem Location Identification Accuracy by Chaos Type', fontsize=16, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=14)
plt.ylim(0, 105)
plt.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar, value in zip(bars, accuracy_percentages):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{value}%', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
